In [1]:
from pathlib import Path
import csv
import re

# =========================
# CẤU HÌNH
# =========================
PROJECT_ROOT = Path(".").resolve()   # đổi thành đường dẫn project nếu cần
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "project_structure"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IGNORE_DIRS = {
    ".git",
    "__pycache__",
    ".ipynb_checkpoints",
    ".venv",
    "venv",
    "env",
    "node_modules"
}

IGNORE_FILES = {
    ".DS_Store",
    "Thumbs.db"
}

MAX_DEPTH = None   # None = quét toàn bộ; hoặc đặt số như 5

# =========================
# HÀM HỖ TRỢ
# =========================
def should_ignore(path: Path) -> bool:
    if path.name in IGNORE_FILES:
        return True
    if any(part in IGNORE_DIRS for part in path.parts):
        return True
    return False

def safe_node_id(path_str: str) -> str:
    return "N_" + re.sub(r"[^a-zA-Z0-9_]", "_", path_str)

def get_file_type(path: Path) -> str:
    if path.is_dir():
        return "directory"
    ext = path.suffix.lower()
    return ext if ext else "no_extension"

def collect_structure(root: Path, max_depth=None):
    items = []

    def walk(current: Path, depth: int):
        if max_depth is not None and depth > max_depth:
            return

        try:
            children = sorted(
                [p for p in current.iterdir() if not should_ignore(p)],
                key=lambda x: (not x.is_dir(), x.name.lower())
            )
        except PermissionError:
            return

        for child in children:
            rel_path = child.relative_to(root)
            items.append({
                "name": child.name,
                "relative_path": str(rel_path).replace("\\", "/"),
                "parent": str(rel_path.parent).replace("\\", "/") if rel_path.parent != Path(".") else "",
                "depth": depth,
                "type": get_file_type(child),
                "is_dir": child.is_dir(),
                "size_bytes": child.stat().st_size if child.is_file() else None
            })
            if child.is_dir():
                walk(child, depth + 1)

    walk(root, 1)
    return items

def build_tree_text(root: Path, max_depth=None) -> str:
    lines = [root.name]

    def walk(current: Path, prefix: str = "", depth: int = 1):
        if max_depth is not None and depth > max_depth:
            return

        try:
            children = sorted(
                [p for p in current.iterdir() if not should_ignore(p)],
                key=lambda x: (not x.is_dir(), x.name.lower())
            )
        except PermissionError:
            return

        total = len(children)
        for idx, child in enumerate(children):
            connector = "└── " if idx == total - 1 else "├── "
            lines.append(prefix + connector + child.name)
            if child.is_dir():
                extension = "    " if idx == total - 1 else "│   "
                walk(child, prefix + extension, depth + 1)

    walk(root)
    return "\n".join(lines)

def build_mermaid(root: Path, items: list[dict]) -> str:
    lines = ["flowchart TD"]

    root_id = safe_node_id(root.name)
    lines.append(f'    {root_id}["📁 {root.name}"]')

    for item in items:
        rel_path = item["relative_path"]
        parent_rel = item["parent"]

        node_id = safe_node_id(rel_path)
        parent_id = root_id if parent_rel == "" else safe_node_id(parent_rel)

        icon = "📁" if item["is_dir"] else "📄"
        label = f'{icon} {item["name"]}'.replace('"', "'")

        lines.append(f'    {node_id}["{label}"]')
        lines.append(f"    {parent_id} --> {node_id}")

    return "\n".join(lines)

def save_inventory_csv(items: list[dict], output_path: Path):
    with output_path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["name", "relative_path", "parent", "depth", "type", "is_dir", "size_bytes"]
        )
        writer.writeheader()
        writer.writerows(items)

# =========================
# CHẠY QUÉT
# =========================
items = collect_structure(PROJECT_ROOT, max_depth=MAX_DEPTH)

tree_text = build_tree_text(PROJECT_ROOT, max_depth=MAX_DEPTH)
mermaid_text = build_mermaid(PROJECT_ROOT, items)

tree_file = OUTPUT_DIR / "project_tree.txt"
csv_file = OUTPUT_DIR / "project_inventory.csv"
mmd_file = OUTPUT_DIR / "project_structure.mmd"
md_file = OUTPUT_DIR / "project_structure.md"

tree_file.write_text(tree_text, encoding="utf-8")
save_inventory_csv(items, csv_file)
mmd_file.write_text(mermaid_text, encoding="utf-8")
md_file.write_text(f"```mermaid\n{mermaid_text}\n```", encoding="utf-8")

print("=" * 80)
print("ĐÃ QUÉT XONG CẤU TRÚC PROJECT")
print("=" * 80)
print(f"Project root: {PROJECT_ROOT}")
print(f"Tổng số mục tìm thấy: {len(items)}")
print()
print("Đã lưu:")
print(f"- Tree text      : {tree_file}")
print(f"- Inventory CSV  : {csv_file}")
print(f"- Mermaid (.mmd) : {mmd_file}")
print(f"- Mermaid (.md)  : {md_file}")
print()
print("XEM NHANH CÂY THƯ MỤC:")
print("-" * 80)
print(tree_text)

ĐÃ QUÉT XONG CẤU TRÚC PROJECT
Project root: C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project
Tổng số mục tìm thấy: 203

Đã lưu:
- Tree text      : C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project\artifacts\project_structure\project_tree.txt
- Inventory CSV  : C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project\artifacts\project_structure\project_inventory.csv
- Mermaid (.mmd) : C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project\artifacts\project_structure\project_structure.mmd
- Mermaid (.md)  : C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project\artifacts\project_structure\project_structure.md

XEM NHANH CÂY THƯ MỤC:
--------------------------------------------------------------------------------
olist_ml_project
├── artifacts
│   ├── data
│   │   ├── candidate_items.csv
│   │   ├── desktop.ini
│   │   ├── known_items.csv
│   │   ├── known_users.csv
│   │   └── product_